# RiskSense AI — Feature Engineering & Risk Scoring

This notebook cleans raw data, computes geospatial features,
builds trend features, and applies the rule-based risk score.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
from src.database import get_connection, query
from src.transform.clean import clean_eonet_events, clean_weather
from src.features.build_features import run as build_features
from src.features.scoring import run as score_risk

In [ ]:
conn = get_connection()
eonet_raw = query(conn, 'SELECT * FROM raw_eonet_events')
weather_raw = query(conn, 'SELECT * FROM raw_weather')
conn.close()

print(f'EONET raw: {len(eonet_raw)} rows')
print(f'Weather raw: {len(weather_raw)} rows')

In [ ]:
eonet_clean = clean_eonet_events(eonet_raw)
weather_clean = clean_weather(weather_raw)
print(f'EONET clean: {len(eonet_clean)} rows')
print(f'Weather clean: {len(weather_clean)} rows')

In [ ]:
feature_df = build_features()
feature_df.head()

In [ ]:
scored_df = score_risk(feature_df)
scored_df[['timestamp', 'risk_score', 'risk_band', 'recommendation']].head(10)

In [ ]:
print('Score distribution:')
print(scored_df['risk_band'].value_counts())
print()
print(f'Mean risk score: {scored_df["risk_score"].mean():.1f}')
print(f'Min: {scored_df["risk_score"].min():.1f}, Max: {scored_df["risk_score"].max():.1f}')